In [ ]:
%pip install selenium
%pip install webdriver-manager
%pip install tqdm

## 모듈화한 함수 불러서 쓰기(각 링크 txt저장까지)

In [13]:
import sys
from pathlib import Path

# 현재 노트북 위치 (AC 폴더)
CURRENT_DIR = Path.cwd()

# 무조건 한 칸 위 (data_crawling)
BASE_DIR = CURRENT_DIR.parent

# import 경로 추가 (맨 앞에 넣는 게 더 안전)
sys.path.insert(0, str(BASE_DIR))

from selenium_auto_module import full_page, product_links


URL = "https://www.lge.co.kr/category/air-conditioners"


html_sample = full_page(URL, "AC")
print(html_sample)

links_sample = product_links("AC")
print(links_sample)

페이지 로딩 완료
1번째 클릭
2번째 클릭
3번째 클릭
4번째 클릭
5번째 클릭
6번째 클릭
7번째 클릭
끝까지 펼침 완료
HTML 길이: 1410434
HTML 저장 완료: AC_full_page_html.txt
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><link rel="preload" as="image" href="/kr/common/footer/sns_youtube.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_insta.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_facebook.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_newsroom.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_kakao.svg"><link rel="preload" as="image"
제품 카드 개수: 234
추출된 링크 개수: 234
저장 완료: AC_product_links.txt
['https://www.lge.co.kr/air-conditioners/fq18gc4ea2', 'https://www.lge.co.kr/air-conditioners/fq18gv4ea2', 'https://www.lge.co.kr/air-conditioners/fq19fv3ed2', 'https://www.lge.co.kr/air-conditioners/sw06adacaj-akorb', 'https://www.lge.co.kr/air-conditioners/fq18gv3ba2']


## 재원추출 및 csv저장

In [16]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 1. 드라이버 실행
# =========================
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 2. 테스트할 에어컨 링크 1개
# =========================
url = "https://www.lge.co.kr/air-conditioners/sq09ek1wes-akor"

driver.get(url)
time.sleep(3)


# =========================
# 3. 공통 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        for label in labels:
            if label in dt_text:
                dd = dt.find_next_sibling("dd")
                if dd:
                    return clean_text(dd.get_text(" ", strip=True))

    return ""


def get_manual_pdf_url(soup):
    for top_area in soup.select("div.top-area"):
        strong = top_area.select_one("strong")
        title = clean_text(strong.get_text()) if strong else ""

        # "제품 사용 설명서(바로보기용)" 제외
        if title == "제품 사용 설명서":
            parent = top_area.find_parent()

            if parent:
                a_tag = parent.select_one("div.btn-group a[href]")

                if a_tag:
                    href = a_tag.get("href", "").strip()

                    if href.startswith("http"):
                        return href
                    elif href.startswith("/"):
                        return "https://www.lge.co.kr" + href
                    else:
                        return href

    return ""


# =========================
# 4. 제품 정보 / 스펙 더보기 클릭
# =========================
try:
    more_btn = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//a[contains(@title,'펼치기') or contains(., '제품 정보 더 보기') or contains(., '제품스펙 상세정보 펼치기')]"
            )
        )
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        more_btn
    )
    time.sleep(1)

    driver.execute_script("arguments[0].click();", more_btn)
    time.sleep(2)

    print("더보기 클릭 완료")

except:
    print("더보기 버튼 없음 또는 이미 펼쳐짐")


# =========================
# 5. HTML 파싱
# =========================
soup = BeautifulSoup(driver.page_source, "html.parser")


# =========================
# 6. 한 행 추출
# =========================
row = {
    "제품명": get_product_name(soup),
    "이미지 링크": get_image_url(soup),
    "가격(원)": get_price(soup),

    "에너지 소비효율등급": get_spec(
        soup,
        "에너지소비효율등급",
        "에너지 소비효율등급",
        "에너지소비효율 등급",
        "에너지 소비효율 등급"
    ),

    "냉방 소비전력(W)": get_spec(
        soup,
        "냉방 소비전력",
        "냉방소비전력"
    ),

    "냉방능력(W)": get_spec(
        soup,
        "냉방능력",
        "냉방 능력"
    ),

    "정격전압(V)": get_spec(
        soup,
        "정격전압",
        "정격 전압"
    ),

    "실외기 크기(mm)": get_spec(
        soup,
        "실외기 크기",
        "실외기크기"
    ),

    "냉방면적(㎡)": get_spec(
        soup,
        "냉방면적",
        "냉방 면적"
    ),

    "바람세기": get_spec(
        soup,
        "바람세기",
        "바람 세기"
    ),

    "제습": get_spec(
        soup,
        "제습"
    ),

    "색상": get_spec(
        soup,
        "색상",
        "컬러",
        "색깔"
    ),

    "제품사용설명서": get_manual_pdf_url(soup),
}


# =========================
# 7. CSV 저장 + 확인
# =========================
df = pd.DataFrame([row])
df.to_csv("lge_ac_one_product.csv", index=False, encoding="utf-8-sig")

print("저장 완료: lge_ac_one_product.csv")
display(df)


# =========================
# 8. 종료
# =========================
driver.quit()

더보기 클릭 완료
저장 완료: lge_ac_one_product.csv


,제품명,이미지 링크,가격(원),에너지 소비효율등급,냉방 소비전력(W),냉방능력(W),정격전압(V),실외기 크기(mm),냉방면적(㎡),바람세기,제습,색상,제품사용설명서
0,LG 휘센 벽걸이에어컨,https://www.lge.co.kr/kr/images/air-conditione...,"1,332,000원",냉방 1등급,870 / 156,"3,600 / 1,150","220, 60",870x650x330,29.3,6단계,O 있음,화이트,https://gscs-b2c.lge.com/open/downloadFile?fil...


In [17]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 1. 드라이버 실행
# =========================
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 2. 에어컨 상품 링크 txt 읽기
# =========================
with open("AC_product_links.txt", "r", encoding="utf-8") as f:
    product_links = [line.strip() for line in f if line.strip()]

print("총 링크 개수:", len(product_links))


# =========================
# 3. 공통 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        for label in labels:
            if label in dt_text:
                dd = dt.find_next_sibling("dd")
                if dd:
                    return clean_text(dd.get_text(" ", strip=True))

    return ""


def get_manual_pdf_url(soup):
    for top_area in soup.select("div.top-area"):
        strong = top_area.select_one("strong")
        title = clean_text(strong.get_text()) if strong else ""

        # "제품 사용 설명서(바로보기용)" 제외
        if title == "제품 사용 설명서":
            parent = top_area.find_parent()

            if parent:
                a_tag = parent.select_one("div.btn-group a[href]")

                if a_tag:
                    href = a_tag.get("href", "").strip()

                    if href.startswith("http"):
                        return href
                    elif href.startswith("/"):
                        return "https://www.lge.co.kr" + href
                    else:
                        return href

    return ""


# =========================
# 4. 전체 링크 순회
# =========================
rows = []
failures = []

pbar = tqdm(product_links, desc="에어컨 상품 크롤링", ncols=120)

for idx, url in enumerate(pbar, start=1):
    try:
        pbar.set_postfix_str(f"{idx}/{len(product_links)} 접속 중")

        driver.get(url)
        time.sleep(3)

        # 제품 정보 / 제품 스펙 더 보기 클릭
        try:
            more_btn = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//a[contains(@title,'펼치기') or contains(., '제품 정보 더 보기') or contains(., '제품스펙 상세정보 펼치기')]"
                    )
                )
            )

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});",
                more_btn
            )
            time.sleep(1)

            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(2)

        except:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")

        row = {
            "제품명": get_product_name(soup),
            "이미지 링크": get_image_url(soup),
            "가격(원)": get_price(soup),

            "에너지 소비효율등급": get_spec(
                soup,
                "에너지소비효율등급",
                "에너지 소비효율등급",
                "에너지소비효율 등급",
                "에너지 소비효율 등급"
            ),

            "냉방 소비전력(W)": get_spec(
                soup,
                "냉방 소비전력",
                "냉방소비전력"
            ),

            "냉방능력(W)": get_spec(
                soup,
                "냉방능력",
                "냉방 능력"
            ),

            "정격전압(V)": get_spec(
                soup,
                "정격전압",
                "정격 전압"
            ),

            "실외기 크기(mm)": get_spec(
                soup,
                "실외기 크기",
                "실외기크기"
            ),

            "냉방면적(㎡)": get_spec(
                soup,
                "냉방면적",
                "냉방 면적"
            ),

            "바람세기": get_spec(
                soup,
                "바람세기",
                "바람 세기"
            ),

            "제습": get_spec(
                soup,
                "제습"
            ),

            "색상": get_spec(
                soup,
                "색상",
                "컬러",
                "색깔"
            ),

            "제품사용설명서": get_manual_pdf_url(soup),
        }

        rows.append(row)

        pbar.set_postfix_str(f"완료: {row['제품명'][:25]}")

        # 10개마다 임시 저장
        if idx % 10 == 0:
            pd.DataFrame(rows).to_csv(
                "lge_ac_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

    except Exception as e:
        failures.append({"url": url, "error": str(e)})
        pbar.set_postfix_str(f"실패: {idx}/{len(product_links)}")


# =========================
# 5. 최종 CSV 저장
# =========================
df = pd.DataFrame(rows)

df.to_csv(
    "lge_ac_all_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 저장 완료: lge_ac_all_products.csv")
print("총 수집 개수:", len(df))
print("실패 개수:", len(failures))

display(df.head())


# =========================
# 6. 실패 목록 저장
# =========================
if failures:
    pd.DataFrame(failures).to_csv(
        "lge_ac_failures.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("실패 목록 저장 완료: lge_ac_failures.csv")


# =========================
# 7. 드라이버 종료
# =========================
driver.quit()

총 링크 개수: 234


에어컨 상품 크롤링: 100%|█████████| 234/234 [1:12:15<00:00, 18.53s/it, 완료: LG 휘센 오브제컬렉션 엣지 창호형 에어컨 마]t, 234/234 접속 중]/s]

전체 저장 완료: lge_ac_all_products.csv
총 수집 개수: 234
실패 개수: 0


,제품명,이미지 링크,가격(원),에너지 소비효율등급,냉방 소비전력(W),냉방능력(W),정격전압(V),실외기 크기(mm),냉방면적(㎡),바람세기,제습,색상,제품사용설명서
0,LG 휘센 오브제컬렉션 타워II 에어컨 2in1 (4시리즈),https://www.lge.co.kr/kr/images/air-conditione...,,2등급,"1,830 / 310","6,500 / 1,600","220, 60",870 x 650 x 330,71.5(52.8+18.7),5단계,O 있음,크림 화이트,https://gscs-b2c.lge.com/open/downloadFile?fil...
1,LG 휘센 AI 오브제컬렉션 뷰I 프로 에어컨 2in1 (6시리즈),https://www.lge.co.kr/kr/images/air-conditione...,"4,107,000원",2등급,"2,100 / 310","7,200 / 1,600","220, 60",870x650x330,71.5(52.8+18.7),5단계,O 있음,에센스 화이트,https://gscs-b2c.lge.com/open/downloadFile?fil...
2,LG 휘센 오브제컬렉션 타워II 에어컨 2in1 (1시리즈),https://www.lge.co.kr/kr/images/air-conditione...,,2등급,"2,070 / 310","7,000 / 1,600","220, 60",870 x 650 x 330,75.6(56.9+18.7),5단계,O 있음,카밍 베이지,https://gscs-b2c.lge.com/open/downloadFile?fil...
3,LG 휘센 오브제컬렉션 타워II 에어컨 2in1 (1시리즈),https://www.lge.co.kr/kr/images/air-conditione...,,2등급,"2,070 / 310","7,000 / 1,600","220, 60",870 x 650 x 330,75.6(56.9+18.7),5단계,O 있음,카밍 베이지,https://gscs-b2c.lge.com/open/downloadFile?fil...
4,LG 휘센 오브제컬렉션 타워II 에어컨 2in1 (1시리즈),https://www.lge.co.kr/kr/images/air-conditione...,,2등급,"2,070 / 310","7,000 / 1,600","220, 60",870 x 650 x 330,75.6(56.9+18.7),5단계,O 있음,크림 화이트,https://gscs-b2c.lge.com/open/downloadFile?fil...


### 실내기 크기 추가

In [ ]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 1. 기존 CSV 읽기
# =========================
df = pd.read_csv("lge_ac_all_products.csv", encoding="utf-8-sig")


# =========================
# 2. 링크 txt 읽기
# =========================
with open("AC_product_links.txt", "r", encoding="utf-8") as f:
    product_links = [line.strip() for line in f if line.strip()]

print("CSV 행 개수:", len(df))
print("링크 개수:", len(product_links))


# =========================
# 3. 드라이버 실행
# =========================
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 4. 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_spec(soup, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        for label in labels:
            if label in dt_text:
                dd = dt.find_next_sibling("dd")
                if dd:
                    return clean_text(dd.get_text(" ", strip=True))

    return ""


# =========================
# 5. 실내기 크기만 추출
# =========================
indoor_sizes = []

for url in tqdm(product_links, desc="실내기 크기 추출", ncols=120):
    try:
        driver.get(url)
        time.sleep(3)

        try:
            more_btn = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//a[contains(@title,'펼치기') or contains(., '제품 정보 더 보기') or contains(., '제품스펙 상세정보 펼치기')]"
                    )
                )
            )

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});",
                more_btn
            )
            time.sleep(1)

            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(2)

        except:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")

        indoor_size = get_spec(
        soup,
        "스탠드 실내기 크기",
        "실내기 크기",
        "실내기크기",
        "실내기 크기_WxHxD",
        "실내기 크기 (WxHxD",
        "실내기크기(WxHxD"
        )   

        indoor_sizes.append(indoor_size)

    except Exception:
        indoor_sizes.append("")


driver.quit()


# =========================
# 6. CSV에 컬럼 삽입
# =========================
if "실내기 크기(mm)" in df.columns:
    df = df.drop(columns=["실내기 크기(mm)"])

insert_idx = df.columns.get_loc("실외기 크기(mm)") + 1

df.insert(
    insert_idx,
    "실내기 크기(mm)",
    indoor_sizes[:len(df)]
)


# =========================
# 7. 저장
# =========================
df.to_csv("lge_ac_all_products_updated.csv", index=False, encoding="utf-8-sig")

print("저장 완료: lge_ac_all_products_updated.csv")
display(df.head())

CSV 행 개수: 234
링크 개수: 234


실내기 크기 추출:   0%|                                                                         | 0/234 [00:00<?, ?it/s]